# VQE Algorithm

Variational Quantum Eigensolvers (VQE) are a leading algorithm for noisy intermediate-scale quantum (NISQ) devices. They combine a parameterized quantum circuit (the *ansatz*) with a classical optimizer to approximate low-lying eigenvalues of a Hamiltonian by minimizing an energy expectation value.

In this notebook, we apply VQE to a two-spin (two-qubit) **Heisenberg XXZ** model and compare the variational estimate of the ground-state energy against an exact classical result.

## Problem Context

The (nearest-neighbor) Heisenberg model describes interactions between **spin-$\tfrac{1}{2}$** degrees of freedom on a lattice. For two sites, we map each spin-$\tfrac{1}{2}$ to a single qubit, so the Hilbert space is $\mathbb{C}^2 \otimes \mathbb{C}^2$.

> **Pauli operators and measurement.** The spin operators are proportional to the Pauli matrices: $S_\alpha = \tfrac{1}{2}\sigma_\alpha$ for $\alpha\in\{x,y,z\}$. Measuring a qubit in the $X$, $Y$, or $Z$ basis corresponds to measuring the observables $\sigma_x$, $\sigma_y$, or $\sigma_z$ (eigenvalues $\pm 1$).

> In VQE we estimate expectation values like $\langle \sigma_x\otimes\sigma_x \rangle$, $\langle \sigma_y\otimes\sigma_y \rangle$, and $\langle \sigma_z\otimes\sigma_z \rangle$ by running the circuit many times and averaging measurement outcomes in the appropriate bases.

### The Hamiltonian we’ll solve
A common two-qubit **XXZ** Heisenberg Hamiltonian is

$$H = J\,(\sigma_x\otimes\sigma_x + \sigma_y\otimes\sigma_y + \Delta\,\sigma_z\otimes\sigma_z),$$

where $J$ sets the interaction strength and $\Delta$ is the anisotropy. (Some conventions use $S_\alpha=\tfrac12\sigma_\alpha$, which changes the overall prefactor by $\tfrac14$; we’ll be explicit in code about which convention we use.)

VQE prepares a trial state $|\psi(\theta)\rangle$ and minimizes the energy

$$E(\theta)=\langle\psi(\theta)|H|\psi(\theta)\rangle,$$

estimated from measurement statistics.

This model is small enough to be solved exactly on a classical computer, allowing direct validation of the VQE results. At the same time, it captures key challenges of variational quantum algorithms, such as ansatz design, optimization landscapes, and measurement noise.

## Import Libraries

In [8]:
# 1) Imports
import numpy as np
from numpy import pi

# Qiskit (core + primitives)
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator
from qiskit.circuit import ParameterVector

# Plotting (optional later)
import matplotlib.pyplot as plt

## Ansatz (parameterized trial state)

The **ansatz** is the parameterized quantum circuit that prepares the trial state $|\psi(\theta)\rangle$. Instead of searching over all possible quantum states (exponentially large), VQE restricts the search to a *family* of states generated by varying the circuit parameters $\theta = (\theta_1, \theta_2, \dots, \theta_n)$.

### Design considerations
- **Expressivity**: Can the ansatz represent (or approximate) the true ground state? If your ansatz is too restrictive, VQE cannot reach the optimal energy.
- **Depth vs. trainability**: Deeper circuits are more expressive but harder to optimize and more susceptible to noise on NISQ hardware.
- **Problem structure**: Good ansätze exploit symmetries or physical intuition (e.g., particle conservation, spin sectors).

### Hardware-Efficient Ansatz (HEA)
We use a simple **hardware-efficient ansatz** with the following structure (repeated for `depth` layers):

1. **Local single-qubit rotations**: $R_y(\theta)$ and $R_z(\theta)$ on each qubit (together they span all single-qubit unitaries)
2. **Entangling layer**: CNOT gates to create correlations between qubits

For **depth=1**, we get **4 parameters** and a single entangling layer—enough flexibility to explore interesting entangled states while remaining shallow enough for near-term devices.

This class of ansätze is called "hardware-efficient" because it uses native gates available on most quantum processors (superconducting qubits, trapped ions, etc.), minimizing compilation overhead.

In [9]:
# 2) Ansatz (parameterized trial state)
def he_ansatz_2q(depth: int = 1):
    """Hardware-efficient ansatz for 2 qubits.

    Structure per layer:
      - Single-qubit rotations: RY then RZ on each qubit
      - Entangler: CX(0 -> 1)

    Returns:
      circuit: parameterized QuantumCircuit
      theta: ParameterVector containing all parameters (length = 4*depth)
    """
    if depth < 1:
        raise ValueError("depth must be >= 1")

    theta = ParameterVector("θ", length=4 * depth)
    qc = QuantumCircuit(2, name=f"HEA2q_d{depth}")

    k = 0
    for _ in range(depth):
        # Local rotations
        qc.ry(theta[k + 0], 0)
        qc.rz(theta[k + 1], 0)
        qc.ry(theta[k + 2], 1)
        qc.rz(theta[k + 3], 1)
        k += 4
        # Entangling layer
        qc.cx(0, 1)

    return qc, theta

## Hamiltonian (the energy operator)

The **Hamiltonian** $H$ is the quantum operator that encodes the energy of the system. In VQE, the Hamiltonian defines the **objective function**: for any quantum state $|\psi\rangle$, the expectation value

$$
E = \langle\psi|H|\psi\rangle
$$

gives the energy of that state. The goal is to find the state that minimizes this energy—the **ground state**.

### Role in VQE
- The **ansatz** prepares trial states $|\psi(\theta)\rangle$ by varying parameters $\theta$
- The **Hamiltonian** scores each state by computing $E(\theta) = \langle\psi(\theta)|H|\psi(\theta)\rangle$
- The **classical optimizer** adjusts $\theta$ to minimize $E(\theta)$

By the **variational principle**, $E(\theta) \geq E_0$ (the true ground-state energy), so minimizing over $\theta$ gives the best approximation within the ansatz's expressivity.

### XXZ Heisenberg Hamiltonian structure
Our two-qubit Hamiltonian is

$$
H = J(\sigma_x\otimes\sigma_x + \sigma_y\otimes\sigma_y + \Delta\,\sigma_z\otimes\sigma_z),
$$

which is a sum of **Pauli strings** (products of Pauli operators acting on different qubits):
- Each term like $\sigma_x\otimes\sigma_x$ measures spin–spin correlation along one axis
- $J$ controls the overall interaction strength
- $\Delta$ is the **anisotropy parameter**: when $\Delta = 1$, the model is isotropic (rotationally symmetric); $\Delta \neq 1$ breaks symmetry along the $z$ axis

### Physical interpretation
- **$J > 0$ (antiferromagnetic)**: favors anti-aligned spins → singlet ground state
- **$J < 0$ (ferromagnetic)**: favors aligned spins → triplet ground state
- **$\Delta$**: tunes whether the system prefers alignment in the $xy$ plane ($\Delta < 1$) or along $z$ ($\Delta > 1$)

### Measurement on quantum hardware
To evaluate $\langle H\rangle$, we measure each Pauli term separately:
1. Prepare $|\psi(\theta)\rangle$ using the ansatz
2. Rotate qubits into the appropriate measurement basis ($X$, $Y$, or $Z$)
3. Measure many times and average the outcomes
4. Combine weighted averages: $\langle H\rangle = J\langle XX\rangle + J\langle YY\rangle + J\Delta\langle ZZ\rangle$

In this notebook, we use `StatevectorEstimator` for exact (noiseless) evaluation, but on real hardware you'd run circuits with finite shots.

In [10]:
# 3) XXZ Heisenberg Hamiltonian
def heisenberg_xxz_2q(J: float = 1.0, delta: float = 1.0) -> SparsePauliOp:
    """Return H = J(XX + YY + delta ZZ) as a SparsePauliOp on 2 qubits."""
    paulis = ["XX", "YY", "ZZ"]
    coeffs = [J, J, J * delta]
    return SparsePauliOp(paulis, coeffs=np.asarray(coeffs, dtype=complex))

# Example Hamiltonian (you can change J and delta later)
J = 1.0
Delta = 1.0
H = heisenberg_xxz_2q(J=J, delta=Delta)
H

SparsePauliOp(['XX', 'YY', 'ZZ'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j])

## Energy Evaluation

Now we connect the ansatz and Hamiltonian to compute the core quantity in VQE: the **energy expectation value** $E(\theta) = \langle\psi(\theta)|H|\psi(\theta)\rangle$.

### The evaluation workflow
1. **Bind parameters**: Take the abstract parameterized circuit (ansatz) and substitute concrete values for $\theta$ to get a specific quantum state $|\psi(\theta)\rangle$
2. **Run measurement**: Use a quantum estimator to evaluate $\langle\psi(\theta)|H|\psi(\theta)\rangle$
   - On real hardware: run the circuit many times with basis rotations for each Pauli term, average measurement outcomes
   - With `StatevectorEstimator`: compute the exact expectation value from the full statevector (no noise, no sampling error)
3. **Return energy**: The scalar result $E(\theta)$ is what the classical optimizer will try to minimize

### Why `StatevectorEstimator`?
For this tutorial, we use **exact statevector simulation** rather than shot-based sampling:
- ✅ **Validation**: eliminates measurement noise, letting us verify VQE's optimization performance cleanly
- ✅ **Speed**: for 2 qubits, exact simulation is faster than shot-based sampling
- ⚠️ **Not scalable**: statevector simulation scales exponentially ($2^n$ amplitudes for $n$ qubits), so it only works for small systems

When moving to larger problems or real quantum hardware, you'd switch to `Sampler` or `Estimator` primitives with finite shots.

### The `energy_expectation` function
This helper wraps the full evaluation pipeline:
```python
energy_expectation(circuit, hamiltonian, param_values) -> float
```
It's the bridge between the quantum ansatz and the classical optimizer—given any $\theta$, it returns the corresponding energy $E(\theta)$.

In [ ]:
# 4) Circuit + energy evaluation helper
estimator = StatevectorEstimator()

def energy_expectation(circuit: QuantumCircuit, hamiltonian: SparsePauliOp, param_values: np.ndarray) -> float:
    """Compute <psi(theta)|H|psi(theta)> using an exact statevector estimator."""
    bound = circuit.assign_parameters(param_values, inplace=False)
    job = estimator.run([(bound, hamiltonian)])
    result = job.result()[0]
    
    return float(np.real(result.data.evs))

# Build a parameterized circuit instance
ansatz, theta = he_ansatz_2q(depth=1)
ansatz.draw(output="text")

┌──────────┐┌──────────┐     
q_0: ┤ Ry(θ[0]) ├┤ Rz(θ[1]) ├──■──
     ├──────────┤├──────────┤┌─┴─┐
q_1: ┤ Ry(θ[2]) ├┤ Rz(θ[3]) ├┤ X ├
     └──────────┘└──────────┘└───┘

## VQE Optimization Loop

Now we implement the classical optimization routine that searches for the optimal parameters $\theta^*$ by minimizing the energy function $E(\theta)$.

### The VQE algorithm
1. **Initialize** parameters $\theta$ (randomly or with a heuristic guess)
2. **Evaluate** energy: compute $E(\theta) = \langle\psi(\theta)|H|\psi(\theta)\rangle$ using the quantum circuit
3. **Optimize**: classical optimizer suggests new $\theta$ to reduce $E(\theta)$
4. **Iterate** steps 2-3 until convergence
5. **Return** optimal parameters $\theta^*$ and ground-state energy estimate $E(\theta^*)$

### Design for flexibility
The `run_vqe` function below is designed to work with **any energy evaluation strategy**:
- Pass in a custom `energy_func` that computes $E(\theta)$ given parameters
- This allows us to later swap between:
  - ✅ Exact statevector simulation (current approach)
  - 🔄 Shot-based simulation with `Sampler`/`Estimator` (mimicking real hardware with noise)
  - 🚀 Execution on real quantum hardware

The optimizer only sees a "black box" function `energy_func(theta) -> float`, so we can change the quantum backend without touching the optimization code.

### Classical optimizer choice
We use **COBYLA** (Constrained Optimization BY Linear Approximation):
- Derivative-free (doesn't need gradients)
- Works well with noisy objective functions
- Standard choice for VQE tutorials

For production, you might explore gradient-based methods (parameter-shift rule) or other optimizers like SPSA, L-BFGS-B, etc.

In [12]:
from scipy.optimize import minimize
from functools import partial

def objective(params, iteration_count, energy_history, ansatz, hamiltonian, energy_func):
    """Objective function: returns energy for given parameters."""
    iteration_count[0] += 1
    energy = energy_func(ansatz, hamiltonian, params)
    energy_history.append(energy)
    
    # Print progress every 20 iterations
    if iteration_count[0] % 20 == 0 or iteration_count[0] == 1:
        print(f"Iteration {iteration_count[0]:3d}: E = {energy:+.6f}")
    
    return energy


def run_vqe(
    ansatz: QuantumCircuit,
    hamiltonian: SparsePauliOp,
    energy_func,
    initial_params: np.ndarray = None,
    method: str = "COBYLA",
    max_iter: int = 200
):
    """
    Run VQE optimization to find ground state energy.
    
    Parameters:
    -----------
    ansatz : QuantumCircuit
        Parameterized quantum circuit (the trial state generator)
    hamiltonian : SparsePauliOp
        The Hamiltonian whose ground state we seek
    energy_func : callable
        Function with signature energy_func(circuit, hamiltonian, params) -> float
        This allows swapping between exact simulation, shot-based simulation, or real hardware
    initial_params : np.ndarray, optional
        Starting parameter values. If None, initialize randomly in [0, 2π)
    method : str
        Classical optimizer to use (default: COBYLA)
    max_iter : int
        Maximum number of optimizer iterations
        
    Returns:
    --------
    result : dict
        - 'optimal_params': best parameters found
        - 'optimal_energy': minimum energy achieved
        - 'num_iterations': total function evaluations
        - 'success': whether optimization converged
        - 'energy_history': list of energies at each iteration
    """
    num_params = ansatz.num_parameters
    
    # Initialize parameters if not provided
    if initial_params is None:
        initial_params = np.random.uniform(0, 2 * np.pi, size=num_params)
    
    # Track optimization progress
    iteration_count = [0]
    energy_history = []
    
    # Create a partial function that binds the extra arguments
    # so minimize only needs to pass the params
    objective_func = partial(
        objective,
        iteration_count=iteration_count,
        energy_history=energy_history,
        ansatz=ansatz,
        hamiltonian=hamiltonian,
        energy_func=energy_func
    )
    
    # Run classical optimization
    print("Starting VQE optimization...")
    print(f"Optimizer: {method}, Max iterations: {max_iter}")
    print(f"Initial energy: {objective_func(initial_params):+.6f}")
    print("-" * 50)
    
    opt_result = minimize(
        objective_func,
        initial_params,
        method=method,
        options={'maxiter': max_iter}
    )
    
    print("-" * 50)
    print(f"Optimization complete!")
    print(f"Final energy: {opt_result.fun:+.6f}")
    print(f"Total iterations: {iteration_count[0]}")
    
    return {
        'optimal_params': opt_result.x,
        'optimal_energy': opt_result.fun,
        'num_iterations': iteration_count[0],
        'success': opt_result.success,
        'energy_history': energy_history
    }

In [14]:
# Run VQE with exact statevector evaluation
print("="*60)
print("VQE for 2-qubit Heisenberg XXZ model")
print(f"Hamiltonian: J={J}, Δ={Delta}")
print("="*60)

vqe_result = run_vqe(
    ansatz=ansatz,
    hamiltonian=H,
    energy_func=energy_expectation,  # Using exact statevector for now
    initial_params=None,  # Random initialization
    method="COBYLA",
    max_iter=200
)

print("\n" + "="*60)
print("VQE Results:")
print(f"Ground state energy estimate: {vqe_result['optimal_energy']:.6f}")
print(f"Optimal parameters: {vqe_result['optimal_params']}")
print("="*60)

VQE for 2-qubit Heisenberg XXZ model
Hamiltonian: J=1.0, Δ=1.0
Starting VQE optimization...
Optimizer: COBYLA, Max iterations: 200
Iteration   1: E = -0.961343
Initial energy: -0.961343
--------------------------------------------------
Iteration  20: E = -2.994401
Iteration  40: E = -2.999975
Iteration  60: E = -3.000000
--------------------------------------------------
Optimization complete!
Final energy: -3.000000
Total iterations: 68

VQE Results:
Ground state energy estimate: -3.000000
Optimal parameters: [1.57076362 3.14162134 3.14162648 5.68334501]


## Shot-based Simulation (Mimicking Real Hardware)

Now we implement a more realistic energy evaluation using **shot-based sampling** instead of exact statevector simulation. This mimics what happens on real quantum hardware:

### Key differences from statevector simulation
- **Finite shots**: We run the circuit a fixed number of times (e.g., 2000 shots) and estimate expectation values from measurement statistics
- **Sampling noise**: Results fluctuate due to finite sampling—more shots give better accuracy but take longer
- **Realistic**: This is how real quantum computers work (they can't access the full statevector)

### How it works
For each Pauli term in the Hamiltonian:
1. Prepare $|\psi(\theta)\rangle$ with the ansatz
2. Apply basis rotations to measure the required Pauli observable
3. Run many shots and count outcomes
4. Estimate $\langle P \rangle$ from the measurement statistics
5. Combine weighted estimates: $E \approx \sum_i c_i \langle P_i \rangle$

### Trade-off: accuracy vs. cost
- More shots → more accurate energy estimates → slower optimization
- Fewer shots → noisier estimates → potentially harder to optimize but faster per iteration

For this demonstration, we'll use **2000 shots** as a reasonable starting point.

In [18]:
from qiskit.primitives import StatevectorSampler

def energy_expectation_shots(
    circuit: QuantumCircuit, 
    hamiltonian: SparsePauliOp, 
    param_values: np.ndarray,
    shots: int = 2000
) -> float:
    """
    Compute <psi(theta)|H|psi(theta)> using shot-based sampling.
    
    This mimics real quantum hardware by:
    1. Running the circuit multiple times (shots)
    2. Measuring in appropriate bases for each Pauli term
    3. Estimating expectation values from measurement statistics
    
    Parameters:
    -----------
    circuit : QuantumCircuit
        Parameterized quantum circuit
    hamiltonian : SparsePauliOp
        Hamiltonian as sum of Pauli strings
    param_values : np.ndarray
        Parameter values to bind to the circuit
    shots : int
        Number of measurement shots per Pauli term (default: 2000)
        
    Returns:
    --------
    energy : float
        Estimated energy expectation value
    """
    # Bind parameters to get concrete circuit
    bound_circuit = circuit.assign_parameters(param_values, inplace=False)
    
    # Initialize sampler (shot-based measurement simulator)
    sampler = StatevectorSampler()
    
    energy = 0.0
    
    # Decompose Hamiltonian into Pauli terms
    # Each term contributes: coefficient * <Pauli_string>
    for pauli_string, coeff in zip(hamiltonian.paulis, hamiltonian.coeffs):
        # Create measurement circuit for this Pauli term
        meas_circuit = bound_circuit.copy()
        
        # Add basis rotations for each qubit based on Pauli operator
        for qubit_idx, pauli_op in enumerate(str(pauli_string)[::-1]):  # Qiskit orders qubits reversed
            if pauli_op == 'X':
                meas_circuit.h(qubit_idx)  # Rotate X basis to Z basis
            elif pauli_op == 'Y':
                meas_circuit.sdg(qubit_idx)  # Rotate Y basis to Z basis
                meas_circuit.h(qubit_idx)
            # Z needs no rotation (computational basis)
        
        # Add measurements
        meas_circuit.measure_all()
        
        # Run circuit with shots (note: need to wrap circuit in list for Qiskit 2.0+)
        job = sampler.run([meas_circuit], shots=shots)
        result = job.result()
        
        # Get measurement counts from BitArray
        counts = result[0].data.meas.get_counts()
        
        # Compute expectation value from measurement statistics
        # For Pauli strings, eigenvalues are ±1 based on parity
        # 
        # Parity explanation:
        # - Count number of 1's in bitstring: '00'→0, '01'→1, '10'→1, '11'→2
        # - Even parity (0 or 2 ones) → eigenvalue +1 (e.g., |00⟩ or |11⟩)
        # - Odd parity (1 one) → eigenvalue -1 (e.g., |01⟩ or |10⟩)
        #
        # Physical meaning for ZZ:
        # - |00⟩, |11⟩: spins aligned (both up or both down) → ZZ eigenvalue = +1
        # - |01⟩, |10⟩: spins anti-aligned (opposite) → ZZ eigenvalue = -1
        #
        # We compute: <P> = Σ (count × eigenvalue) / total_shots
        expectation = 0.0
        for bitstring, count in counts.items():
            # Count number of 1's in the bitstring
            num_ones = bitstring.count('1')
            parity = num_ones % 2
            
            # Even parity → +1, odd parity → -1
            eigenvalue = 1 if parity == 0 else -1
            expectation += count * eigenvalue
        
        expectation /= shots  # Normalize by total shots
        
        # Add weighted contribution to total energy
        energy += float(np.real(coeff)) * expectation
    
    return energy

# Test with a single energy evaluation
print("Testing shot-based energy evaluation...")
test_params = np.random.uniform(0, 2*np.pi, size=4)
energy_exact = energy_expectation(ansatz, H, test_params)
energy_shots = energy_expectation_shots(ansatz, H, test_params, shots=2000)

print(f"Exact (statevector):  {energy_exact:+.6f}")
print(f"Shots (2000 samples): {energy_shots:+.6f}")
print(f"Difference:           {abs(energy_exact - energy_shots):.6f}")

Testing shot-based energy evaluation...
Exact (statevector):  +0.286531
Shots (2000 samples): +0.330000
Difference:           0.043469


In [19]:
# Run VQE with shot-based simulation
print("\n" + "="*60)
print("VQE with Shot-based Simulation (2000 shots)")
print(f"Hamiltonian: J={J}, Δ={Delta}")
print("="*60)

vqe_result_shots = run_vqe(
    ansatz=ansatz,
    hamiltonian=H,
    energy_func=lambda c, h, p: energy_expectation_shots(c, h, p, shots=2000),
    initial_params=None,  # Random initialization
    method="COBYLA",
    max_iter=200
)

print("\n" + "="*60)
print("VQE Results (Shot-based):")
print(f"Ground state energy estimate: {vqe_result_shots['optimal_energy']:.6f}")
print(f"Optimal parameters: {vqe_result_shots['optimal_params']}")
print("="*60)

# Compare with exact result
print("\n" + "="*60)
print("Comparison:")
print(f"Exact (statevector):  {vqe_result['optimal_energy']:.6f}")
print(f"Shots (2000 samples): {vqe_result_shots['optimal_energy']:.6f}")
print(f"Difference:           {abs(vqe_result['optimal_energy'] - vqe_result_shots['optimal_energy']):.6f}")
print("="*60)


VQE with Shot-based Simulation (2000 shots)
Hamiltonian: J=1.0, Δ=1.0
Starting VQE optimization...
Optimizer: COBYLA, Max iterations: 200
Iteration   1: E = +0.071000
Initial energy: +0.071000
--------------------------------------------------
Iteration  20: E = -2.933000
Iteration  40: E = -2.987000
Iteration  60: E = -2.984000
--------------------------------------------------
Optimization complete!
Final energy: -2.995000
Total iterations: 60

VQE Results (Shot-based):
Ground state energy estimate: -2.995000
Optimal parameters: [1.66801462 3.07866886 3.12786817 4.35403939]

Comparison:
Exact (statevector):  -3.000000
Shots (2000 samples): -2.995000
Difference:           0.005000
Iteration  40: E = -2.987000
Iteration  60: E = -2.984000
--------------------------------------------------
Optimization complete!
Final energy: -2.995000
Total iterations: 60

VQE Results (Shot-based):
Ground state energy estimate: -2.995000
Optimal parameters: [1.66801462 3.07866886 3.12786817 4.35403939

## Real Quantum Hardware Execution

Now we're ready to run VQE on **real IBM Quantum hardware**! This is the ultimate test—running on actual superconducting qubits with all the noise, decoherence, and imperfections of physical quantum systems.

### What changes when moving to real hardware?

1. **Authentication**: Need IBM Quantum API token to access cloud quantum computers
2. **Backend selection**: Choose a specific quantum processor (e.g., `ibm_brisbane`, `ibm_kyoto`)
3. **Queue time**: Jobs wait in a queue before execution (can take minutes to hours depending on demand)
4. **Hardware noise**: Real decoherence, gate errors, readout errors affect results
5. **Transpilation**: Circuit must be compiled to match hardware topology and native gate set
6. **Cost awareness**: Real QPU time is limited (check your IBM Quantum plan)

### The workflow

1. **Connect** to IBM Quantum using your API token
2. **Select backend**: Choose least-busy available quantum computer
3. **Transpile circuit**: Optimize for hardware constraints
4. **Create Estimator**: Use `BackendEstimatorV2` instead of `StatevectorEstimator`
5. **Run VQE**: Same optimization loop, but with real quantum hardware evaluations!

### Important notes

- **Start with fewer iterations** (e.g., 20-50) to avoid long queue times
- **Monitor your usage**: Check IBM Quantum dashboard for remaining runtime
- **Be patient**: Queue times vary; consider running during off-peak hours
- **Save results**: Jobs can be retrieved later using job ID if notebook crashes

Let's set it up!

In [ ]:
# IBM Quantum Setup and Backend Selection
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as RuntimeEstimator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# ============================================================================
# STEP 1: Authenticate with IBM Quantum
# ============================================================================
# IMPORTANT: Replace 'YOUR_IBM_QUANTUM_API_TOKEN_HERE' with your actual API token
# Get your token from: https://quantum.ibm.com/account
IBM_API_TOKEN = 'your_api_token_here'

# Save credentials (only needed once - comment out after first run)
QiskitRuntimeService.save_account(channel="ibm_quantum_platform", token=IBM_API_TOKEN, overwrite=True)

# Load service (use this for subsequent runs)
service = QiskitRuntimeService(channel="ibm_quantum_platform")

print("✓ Connected to IBM Quantum!")
print(f"Available backends: {len(service.backends())} quantum computers")

# ============================================================================
# STEP 2: Select a Backend (Quantum Computer)
# ============================================================================
# Option A: Get the least busy backend with at least 2 qubits
backend = service.least_busy(simulator=False, operational=True, min_num_qubits=2)

# Option B: Specify a particular backend by name (uncomment to use)
# backend = service.backend('ibm_brisbane')  # or 'ibm_kyoto', 'ibm_osaka', etc.

print(f"\n🖥️  Selected backend: {backend.name}")
print(f"   Number of qubits: {backend.num_qubits}")
print(f"   Pending jobs: {backend.status().pending_jobs}")
print(f"   Backend version: {backend.version}")

# ============================================================================
# STEP 3: Transpile Circuit for Hardware
# ============================================================================
# The ansatz circuit needs to be compiled to match hardware constraints
# (qubit connectivity, native gates, etc.)

# Create a pass manager for transpilation (optimization_level=3 for best results)
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

# Transpile the ansatz (note: parameters will be bound later during VQE)
transpiled_ansatz = pm.run(ansatz)

print(f"\n🔧 Transpiled circuit:")
print(f"   Original depth: {ansatz.depth()}")
print(f"   Transpiled depth: {transpiled_ansatz.depth()}")
print(f"   Original gates: {ansatz.count_ops()}")
print(f"   Transpiled gates: {transpiled_ansatz.count_ops()}")

# ============================================================================
# STEP 3b: Transpile Hamiltonian to Match Physical Qubits
# ============================================================================
# The transpiled circuit uses physical qubits on the hardware, so we need to
# map the Hamiltonian to the same physical qubits

# Get the qubit mapping from logical (0, 1) to physical qubits
initial_layout = transpiled_ansatz.layout.initial_layout
# Extract the physical qubit indices (initial_layout returns Qubit objects, we need integers)
physical_qubits = [initial_layout[i]._index for i in range(2)]

print(f"\n📍 Qubit mapping:")
print(f"   Logical qubit 0 → Physical qubit {physical_qubits[0]}")
print(f"   Logical qubit 1 → Physical qubit {physical_qubits[1]}")

# Create new Hamiltonian that operates on the physical qubits
# We need to expand the Pauli strings to the full hardware size
def transpile_hamiltonian(hamiltonian, physical_qubits, num_qubits_hardware):
    """
    Expand a Hamiltonian to match hardware qubit count.
    
    Example: "XY" on logical qubits [0,1] mapped to physical [3,5] on 127-qubit system
             becomes "IIIXIYIII...I" (127 qubits total, X at position 3, Y at position 5)
    """
    new_paulis = []
    new_coeffs = []
    
    for pauli_str, coeff in zip(hamiltonian.paulis, hamiltonian.coeffs):
        # Create identity string for all hardware qubits
        expanded_str = ['I'] * num_qubits_hardware
        
        # Map each operator to its physical qubit location
        for logical_idx, pauli_op in enumerate(str(pauli_str)[::-1]):
            physical_idx = physical_qubits[logical_idx]
            expanded_str[physical_idx] = pauli_op
        
        # Qiskit orders qubits reversed, so reverse the string
        expanded_pauli_str = ''.join(expanded_str[::-1])
        new_paulis.append(expanded_pauli_str)
        new_coeffs.append(coeff)
    
    return SparsePauliOp(new_paulis, coeffs=np.array(new_coeffs))

# Create the transpiled Hamiltonian
H_transpiled = transpile_hamiltonian(H, physical_qubits, backend.num_qubits)

print(f"\n🔧 Transpiled Hamiltonian:")
print(f"   Original: {H.paulis}")
print(f"   Expanded to {backend.num_qubits} qubits for hardware")
print(f"   (Operating on physical qubits {physical_qubits})")

# ============================================================================
# STEP 4: Create Hardware Energy Evaluation Function
# ============================================================================
def energy_expectation_hardware(
    circuit: QuantumCircuit,
    hamiltonian: SparsePauliOp,
    param_values: np.ndarray,
    backend,
    shots: int = 1024
) -> float:
    """
    Compute energy expectation value using REAL quantum hardware via IBM Quantum.
    
    This function:
    1. Binds parameter values to the circuit
    2. Uses IBM's RuntimeEstimator to evaluate <H> on real QPU
    3. Returns the energy (includes real hardware noise!)
    
    Parameters:
    -----------
    circuit : QuantumCircuit
        Parameterized quantum circuit (should be pre-transpiled)
    hamiltonian : SparsePauliOp
        Hamiltonian operator
    param_values : np.ndarray
        Parameter values to evaluate
    backend : IBMBackend
        The quantum computer backend
    shots : int
        Number of measurement shots (default: 1024)
        
    Returns:
    --------
    energy : float
        Measured energy expectation value from real hardware
    """
    # Create estimator with hardware backend (pass backend as positional argument)
    estimator = RuntimeEstimator(backend)
    
    # Set options (shots, error mitigation, etc.)
    estimator.options.default_shots = shots
    estimator.options.resilience_level = 1  # Enable basic error mitigation
    
    # Bind parameters
    bound_circuit = circuit.assign_parameters(param_values)
    
    # Run job on quantum hardware
    # Note: This creates a PUB (Primitive Unified Bloc) - the new Qiskit Runtime API format
    job = estimator.run([(bound_circuit, hamiltonian)])
    
    # Wait for job to complete and get result
    result = job.result()
    
    # Extract energy value
    energy = result[0].data.evs
    
    return float(np.real(energy))

print("\n✓ Hardware energy evaluation function ready!")
print("⚠️  Note: Each VQE iteration will submit a job to the quantum computer")

management.get:WARNING:2025-12-27 20:52:29,653: Loading default saved account
qiskit_runtime_service.__init__:WARNING:2025-12-27 20:52:32,472: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2025-12-27 20:52:32,474: Loading instance: open-instance, plan: open
qiskit_runtime_service.__init__:WARNING:2025-12-27 20:52:32,472: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specif

✓ Connected to IBM Quantum!
Available backends: 3 quantum computers
Available backends: 3 quantum computers


qiskit_runtime_service.backends:WARNING:2025-12-27 20:52:34,026: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2025-12-27 20:52:35,130: Using instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2025-12-27 20:52:35,130: Using instance: open-instance, plan: open



🖥️  Selected backend: ibm_fez
   Number of qubits: 156
   Number of qubits: 156
   Pending jobs: 0
   Backend version: 2

🔧 Transpiled circuit:
   Original depth: 3
   Transpiled depth: 10
   Original gates: OrderedDict({'ry': 2, 'rz': 2, 'cx': 1})
   Transpiled gates: OrderedDict({'rz': 8, 'sx': 6, 'cz': 1})

📍 Qubit mapping:
   Logical qubit 0 → Physical qubit 0
   Logical qubit 1 → Physical qubit 1

🔧 Transpiled Hamiltonian:
   Original: ['XX', 'YY', 'ZZ']
   Expanded to 156 qubits for hardware
   (Operating on physical qubits [0, 1])

✓ Hardware energy evaluation function ready!
⚠️  Note: Each VQE iteration will submit a job to the quantum computer
   Pending jobs: 0
   Backend version: 2

🔧 Transpiled circuit:
   Original depth: 3
   Transpiled depth: 10
   Original gates: OrderedDict({'ry': 2, 'rz': 2, 'cx': 1})
   Transpiled gates: OrderedDict({'rz': 8, 'sx': 6, 'cz': 1})

📍 Qubit mapping:
   Logical qubit 0 → Physical qubit 0
   Logical qubit 1 → Physical qubit 1

🔧 Transpiled

In [29]:
# ============================================================================
# RUN VQE ON REAL QUANTUM HARDWARE! 🚀
# ============================================================================

print("="*70)
print(" 🌌 VQE on REAL IBM Quantum Hardware 🌌")
print(f" Backend: {backend.name}")
print(f" Hamiltonian: J={J}, Δ={Delta}")
print("="*70)
print("\n⚠️  IMPORTANT:")
print("   - Start with FEW iterations (20-50) due to queue times")
print("   - Each iteration submits a job to the quantum computer")
print("   - Jobs wait in queue before execution")
print("   - Monitor IBM Quantum dashboard for job status")
print("   - Save results frequently!")
print("\n" + "="*70 + "\n")

# Use the BEST parameters found from statevector simulation as initial guess
# This reduces QPU time needed (starting from a good point)
initial_params_hardware = vqe_result['optimal_params']

print(f"🎯 Using optimized parameters from simulation as starting point:")
print(f"   Initial params: {initial_params_hardware}")
print(f"   Expected energy: ~{vqe_result['optimal_energy']:.4f}")
print()

# Run VQE with real quantum hardware
# NOTE: Set max_iter LOW (20-50) for first test to avoid long queue times!
vqe_result_hardware = run_vqe(
    ansatz=transpiled_ansatz,  # Use transpiled circuit!
    hamiltonian=H_transpiled,  # Use transpiled Hamiltonian (mapped to physical qubits)!
    energy_func=lambda c, h, p: energy_expectation_hardware(
        c, h, p, backend=backend, shots=1024
    ),
    initial_params=initial_params_hardware,  # Start from simulation result
    method="COBYLA",
    max_iter=20  # ⚠️ START SMALL! Increase after successful test
)

print("\n" + "="*70)
print("🎉 VQE Results from Real Quantum Hardware:")
print(f"   Ground state energy: {vqe_result_hardware['optimal_energy']:.6f}")
print(f"   Optimal parameters: {vqe_result_hardware['optimal_params']}")
print(f"   Iterations completed: {vqe_result_hardware['num_iterations']}")
print(f"   Optimization success: {vqe_result_hardware['success']}")
print("="*70)

# ============================================================================
# COMPARISON: Simulation vs Real Hardware
# ============================================================================
print("\n" + "="*70)
print("📊 COMPREHENSIVE COMPARISON")
print("="*70)
print(f"{'Method':<25} {'Energy':<15} {'Difference from Exact'}")
print("-"*70)
print(f"{'Exact (statevector)':<25} {vqe_result['optimal_energy']:>+.6f}     {'[reference]':<15}")
print(f"{'Shot simulation (2000)':<25} {vqe_result_shots['optimal_energy']:>+.6f}     {abs(vqe_result_shots['optimal_energy'] - vqe_result['optimal_energy']):>+.6f}")
print(f"{'Real hardware (1024)':<25} {vqe_result_hardware['optimal_energy']:>+.6f}     {abs(vqe_result_hardware['optimal_energy'] - vqe_result['optimal_energy']):>+.6f}")
print("="*70)

# Additional analysis
hardware_error = abs(vqe_result_hardware['optimal_energy'] - vqe_result['optimal_energy'])
simulation_error = abs(vqe_result_shots['optimal_energy'] - vqe_result['optimal_energy'])

print(f"\n💡 Insights:")
print(f"   • Hardware noise adds ~{(hardware_error / abs(vqe_result['optimal_energy']) * 100):.1f}% error")
print(f"   • Simulation noise adds ~{(simulation_error / abs(vqe_result['optimal_energy']) * 100):.1f}% error")
print(f"   • Hardware is {'more' if hardware_error > simulation_error else 'less'} noisy than shot simulation")
print(f"   • Total QPU iterations: {vqe_result_hardware['num_iterations']}")

print(f"\n✅ Successfully ran VQE on {backend.name}!")
print(f"   Check your IBM Quantum dashboard for detailed job history")
print("="*70)

 🌌 VQE on REAL IBM Quantum Hardware 🌌
 Backend: ibm_fez
 Hamiltonian: J=1.0, Δ=1.0

⚠️  IMPORTANT:
   - Start with FEW iterations (20-50) due to queue times
   - Each iteration submits a job to the quantum computer
   - Jobs wait in queue before execution
   - Monitor IBM Quantum dashboard for job status
   - Save results frequently!


🎯 Using optimized parameters from simulation as starting point:
   Initial params: [1.57076362 3.14162134 3.14162648 5.68334501]
   Expected energy: ~-3.0000

Starting VQE optimization...
Optimizer: COBYLA, Max iterations: 20
Iteration   1: E = +1.085733
Initial energy: +1.085733
--------------------------------------------------
Iteration   1: E = +1.085733
Initial energy: +1.085733
--------------------------------------------------
Iteration  20: E = +1.026827
Iteration  20: E = +1.026827
--------------------------------------------------
Optimization complete!
Final energy: +0.929979
Total iterations: 21

🎉 VQE Results from Real Quantum Hardware:
   G